# DCPA 语言模型能力与架构诊断（只读，不训练）

在一个已有的训练 checkpoint 上跑前向诊断，回答五个问题：

| 测试 | 回答的问题 |
|---|---|
| A 因果泄漏 | 改动未来字节会不会影响之前位置的预测？（不通过则 bpb 无意义） |
| B 路径消融 | MoE 专家、概念路径（`f_out` / `delta_syn`）各自贡献多少 bpb？ |
| C 上下文曲线 | 给模型 N 字节历史，bpb 随 N 怎么变？有效上下文有多长？ |
| D 段内位置 | 段末字节是否明显更难？`p_seg_end` 与 SGSC 脉冲是否一致？ |
| E 参照梯子 | 模型 bpb 相对 n-gram 和通用压缩器处在什么位置？ |

用法：按顺序运行全部单元。只读取 Drive 上的 checkpoint 和语料归档，唯一的写入是 `DCPA_Project/logs/diag_step_*.json`。
可以在训练进行时另开一个运行时跑；任何 GPU 都行，48M 在 T4 上也只要几分钟到十几分钟。

In [ ]:
# Cell 1 [诊断配置 + 挂载 Google Drive]
# ==============================================================================
MODEL_PRESET = "48m"     # "48m" -> dcpa_corpus_v1.tar.gz；"210m" -> dcpa_corpus_v2_cognitive.tar.gz
CKPT_NAME = "auto"       # "auto" = Drive checkpoints/ 下最新的 .pt；也可填 "best_model.pt"、
                         # "milestone_step_015000.pt" 或任意绝对路径（.pt / .safetensors）
EVAL_STAGE = "auto"      # "auto" = 按 checkpoint 的 stage_id 映射 (1->A, 2->B, 3->C, 4+->D)；可强制 "B"/"C"/"D"
EVAL_T = 2048            # 每篇评估文档取开头多少字节（文档不够长会自动降到 1024/512/256）
EVAL_DOCS = 64           # 主评估用多少篇留出文档
BATCH_DOCS = 16          # 每次前向多少篇
CTX_WINDOW = 256         # 测试 C：被打分的目标窗口字节数
NGRAM_TRAIN_MB = 32      # 测试 E：n-gram 用多少 MB 训练语料计数
COMPRESS_MB = 16         # 测试 E：压缩器在多少 MB 留出语料上测
SEED = 1234
# ==============================================================================

import json
import os
import sys
import time
from pathlib import Path

LOCAL_MODE = os.environ.get("DCPA_DIAG_LOCAL", "0") == "1"   # 非 Colab 环境调试用
if MODEL_PRESET not in ("48m", "210m"):
    raise ValueError(f"MODEL_PRESET must be '48m' or '210m', got {MODEL_PRESET!r}")

if not LOCAL_MODE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path(os.environ.get("DCPA_DIAG_DRIVE_ROOT", "/content/drive/MyDrive/DCPA_Project"))
DRIVE_CKPTS = DRIVE_ROOT / "checkpoints"
DRIVE_EXPORT = DRIVE_ROOT / "export"
DRIVE_LOGS = DRIVE_ROOT / "logs"
DRIVE_ARCHIVE = DRIVE_ROOT / "data_archive" / (
    "dcpa_corpus_v1.tar.gz" if MODEL_PRESET == "48m" else "dcpa_corpus_v2_cognitive.tar.gz"
)
NVME_CACHE = Path(os.environ.get("DCPA_DIAG_CACHE", "/content/nvme_cache"))
DATA_DIR = Path(os.environ.get("DCPA_DIAG_DATA_DIR", str(NVME_CACHE / "data")))
DIAG_DIR = NVME_CACHE / "diag"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

RESULTS: dict = {"preset": MODEL_PRESET}
print(f"[Diag] DRIVE_ROOT={DRIVE_ROOT} | DATA_DIR={DATA_DIR} | LOCAL_MODE={LOCAL_MODE}")

In [ ]:
# Cell 2 [部署代码（与训练 notebook 使用同一个 dcpa_colab_bundle.zip）]
import shutil
import subprocess
import zipfile

if LOCAL_MODE:
    CODE_DIR = Path(os.environ["DCPA_DIAG_REPO"])
else:
    BUNDLE_ZIP = DRIVE_ROOT / "code" / "dcpa_colab_bundle.zip"
    CODE_DIR = Path("/content/dcpa")
    if not BUNDLE_ZIP.exists():
        raise FileNotFoundError(f"Missing code bundle at {BUNDLE_ZIP} (run scripts/pack_for_colab.sh and upload).")
    if CODE_DIR.exists():
        shutil.rmtree(CODE_DIR)
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BUNDLE_ZIP, "r") as zf:
        zf.extractall(CODE_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "safetensors", "pyyaml"], check=True)

os.chdir(CODE_DIR)
for p in (str(CODE_DIR), str(CODE_DIR / "dcpa-train-py")):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = DEVICE.type == "cuda"
print(f"[Deploy] code={CODE_DIR} | device={DEVICE} | torch={torch.__version__} | bf16_autocast={USE_BF16}")
if DEVICE.type == "cuda":
    print(f"[Deploy] GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3 [准备留出语料 val.bin / val.idx 与 n-gram 训练样本（优先复用本机 NVMe，否则从 Drive 归档流式读取）]
import tarfile

VAL_BIN = DATA_DIR / "val.bin"
VAL_IDX = DATA_DIR / "val.idx"
TRAIN_BIN = DATA_DIR / "train.bin"
NGRAM_SAMPLE = DIAG_DIR / "ngram_train_sample.bin"
CHUNK = 1 << 20
DATA_DIR.mkdir(parents=True, exist_ok=True)


def _sample_train_from_disk(path: Path, quota_mb: int) -> bytes:
    # Evenly spaced 1 MB chunks so every domain in train.bin is represented.
    mm = np.memmap(path, dtype=np.uint8, mode="r")
    n_chunks_total = max(1, len(mm) // CHUNK)
    take = min(int(quota_mb), n_chunks_total)
    picks = np.linspace(0, n_chunks_total - 1, num=take).astype(np.int64)
    out = b"".join(bytes(mm[int(c) * CHUNK : (int(c) + 1) * CHUNK]) for c in picks)
    del mm
    return out


t0 = time.perf_counter()
need_val = not (VAL_BIN.exists() and VAL_IDX.exists())
need_sample = not NGRAM_SAMPLE.exists()

if need_sample and TRAIN_BIN.exists():
    NGRAM_SAMPLE.write_bytes(_sample_train_from_disk(TRAIN_BIN, NGRAM_TRAIN_MB))
    need_sample = False
    print(f"[Data] n-gram sample taken from local {TRAIN_BIN}")

if need_val or need_sample:
    if not DRIVE_ARCHIVE.exists():
        raise FileNotFoundError(
            f"Neither local corpus under {DATA_DIR} nor Drive archive {DRIVE_ARCHIVE} found. "
            "Check MODEL_PRESET, or run the training notebook's Cell 3 once."
        )
    print(f"[Data] Streaming {DRIVE_ARCHIVE} (reads val.* and a strided sample of train.bin; nothing is written to Drive)...")
    with tarfile.open(DRIVE_ARCHIVE, "r|gz") as tf:
        for member in tf:
            base = Path(member.name).name
            if base in ("val.bin", "val.idx") and need_val:
                src = tf.extractfile(member)
                assert src is not None
                with open(DATA_DIR / base, "wb") as dst:
                    shutil.copyfileobj(src, dst, length=CHUNK)
                print(f"[Data]   extracted {base} ({member.size / 1e6:.1f} MB)")
            elif base == "train.bin" and need_sample:
                src = tf.extractfile(member)
                assert src is not None
                n_chunks_total = max(1, member.size // CHUNK)
                stride = max(1, n_chunks_total // max(1, NGRAM_TRAIN_MB))
                kept = []
                i = 0
                while True:
                    blk = src.read(CHUNK)
                    if not blk:
                        break
                    if i % stride == 0 and len(kept) < NGRAM_TRAIN_MB:
                        kept.append(blk)
                    i += 1
                NGRAM_SAMPLE.write_bytes(b"".join(kept))
                need_sample = False
                print(f"[Data]   sampled {len(kept)} MB of train.bin (1 of every {stride} MB)")

if not (VAL_BIN.exists() and VAL_IDX.exists()):
    raise FileNotFoundError(f"val.bin / val.idx still missing under {DATA_DIR}")

# Exact 16-byte .idx record layout (mirrors data/ragged_dataloader.py IDX_DTYPE)
IDX_DTYPE = np.dtype(
    [("offset", "<u8"), ("length", "<u4"), ("lang_id", "<u1"), ("source_id", "<u1"), ("max_depth", "<u1"), ("style_id", "<u1")]
)
val_idx = np.fromfile(VAL_IDX, dtype=IDX_DTYPE)
val_mm = np.memmap(VAL_BIN, dtype=np.uint8, mode="r")
print(
    f"[Data] val.bin={len(val_mm) / 1e6:.1f} MB | docs={len(val_idx):,} | "
    f"doc length p50/p90/max = {int(np.percentile(val_idx['length'], 50))}/"
    f"{int(np.percentile(val_idx['length'], 90))}/{int(val_idx['length'].max())} B | "
    f"n-gram sample={NGRAM_SAMPLE.stat().st_size / 1e6 if NGRAM_SAMPLE.exists() else 0:.1f} MB | "
    f"{time.perf_counter() - t0:.1f}s"
)

In [ ]:
# Cell 4 [加载 checkpoint 与模型]
import dataclasses

from dcpa.compile_utils import strip_orig_mod_prefix
from dcpa.config import DCPAConfig
from dcpa.model import DCPAModel


def _candidate_ckpts() -> list[Path]:
    env_ckpt = os.environ.get("DCPA_DIAG_CKPT")
    if env_ckpt:
        return [Path(env_ckpt)]
    if CKPT_NAME != "auto":
        p = Path(CKPT_NAME)
        return [p if p.is_absolute() else DRIVE_CKPTS / CKPT_NAME]
    cands = sorted(DRIVE_CKPTS.glob("*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
    final_st = DRIVE_EXPORT / "dcpa_final.safetensors"
    if final_st.exists():
        cands.append(final_st)
    return cands


def _load_any(path: Path):
    if path.suffix == ".safetensors":
        from train.train_stage5_sft import load_dcpa_model_from_weights

        return load_dcpa_model_from_weights(path, device="cpu"), {"step": -1, "stage_id": 4}
    ck = torch.load(str(path), map_location="cpu", weights_only=False)
    cfg_d = dict(ck["config"])
    known = {f.name for f in dataclasses.fields(DCPAConfig)}
    dropped = sorted(set(cfg_d) - known)
    if dropped:
        print(f"[Ckpt][WARN] config keys unknown to this code bundle were ignored: {dropped}")
    mdl = DCPAModel(DCPAConfig(**{k: v for k, v in cfg_d.items() if k in known}))
    sd = strip_orig_mod_prefix(ck["model_state_dict"])
    try:
        mdl.load_state_dict(sd, strict=True)
    except RuntimeError:
        missing, unexpected = mdl.load_state_dict(sd, strict=False)
        print(
            "[Ckpt][WARN] state_dict does not match this code bundle exactly "
            f"(missing={list(missing)[:8]}, unexpected={list(unexpected)[:8]}). "
            "The bundle on Drive is probably newer/older than the code that produced the checkpoint; "
            "results are only trustworthy if these lists are empty or cosmetic."
        )
    return mdl, {"step": int(ck.get("step", -1)), "stage_id": int(ck.get("stage_id", -1))}


cands = _candidate_ckpts()
if not cands:
    raise FileNotFoundError(f"No checkpoint found under {DRIVE_CKPTS} or {DRIVE_EXPORT}")
print("[Ckpt] Candidates (newest first):")
for p in cands:
    if p.exists():
        print(f"    {p.name:32s} {p.stat().st_size / 1e6:8.1f} MB  mtime={time.strftime('%m-%d %H:%M', time.localtime(p.stat().st_mtime))}")

model, ckpt_meta, ckpt_path = None, None, None
for p in cands:
    if not p.exists():
        continue
    try:
        local_p = p
        if not LOCAL_MODE:
            local_p = DIAG_DIR / p.name
            shutil.copy2(p, local_p)  # read from local disk; a slot being overwritten by training fails cleanly below
            if p.suffix == ".safetensors" and (p.parent / "config.json").exists():
                shutil.copy2(p.parent / "config.json", DIAG_DIR / "config.json")
        model, ckpt_meta = _load_any(local_p)
        ckpt_path = p
        break
    except Exception as exc:  # e.g. slot file mid-write
        print(f"[Ckpt][WARN] could not load {p.name}: {type(exc).__name__}: {exc}")
if model is None:
    raise RuntimeError("No loadable checkpoint.")

model = model.to(DEVICE).eval()
cfg = model.cfg
assert model.bcg_pc.cfg is cfg, "ablation by cfg.l_max relies on a shared config object"

_stage_map = {1: "A", 2: "B", 3: "C"}
STAGE = EVAL_STAGE if EVAL_STAGE != "auto" else _stage_map.get(ckpt_meta["stage_id"], "D")
n_total = sum(p.numel() for p in model.parameters())
n_moe = sum(p.numel() for p in model.bcg_pc.moe.experts.parameters())
RESULTS.update(
    ckpt=ckpt_path.name, step=ckpt_meta["step"], ckpt_stage_id=ckpt_meta["stage_id"], eval_stage=STAGE,
    params_total=n_total, params_moe_experts=n_moe,
)
print(
    f"[Ckpt] Loaded {ckpt_path.name} | step={ckpt_meta['step']} | stage_id={ckpt_meta['stage_id']} -> eval stage '{STAGE}' | "
    f"params={n_total / 1e6:.2f}M (MoE experts {n_moe / 1e6:.2f}M = {100 * n_moe / n_total:.1f}%) | "
    f"d_s={cfg.d_s} D={cfg.d_concept} experts={cfg.num_experts} l_max={cfg.l_max}"
)
if STAGE == "A":
    print(
        "[Ckpt][NOTE] Stage A decodes each segment from its OWN encoding (auto-encoding), so its bpb is NOT a "
        "language-model score and tests B/C say little. Re-run once training has reached Stage 2+."
    )

In [ ]:
# Cell 5 [评估工具函数 + 基线 bpb]
import inspect
from contextlib import contextmanager

LN2 = float(np.log(2.0))
FWD_PARAMS = set(inspect.signature(model.forward).parameters)
_DECODE_SIG = inspect.signature(model.pmd.decode_teacher_forced)


def utf8_start(buf: np.ndarray, pos: int) -> int:
    # Move forward to the first byte that is not a UTF-8 continuation byte.
    while pos < len(buf) and (int(buf[pos]) & 0xC0) == 0x80:
        pos += 1
    return pos


def pick_docs(min_len: int, n: int, seed: int) -> np.ndarray:
    ok = np.nonzero(val_idx["length"] >= min_len)[0]
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(ok, size=min(n, len(ok)), replace=False)) if len(ok) else ok


def doc_bytes(doc_i: int) -> np.ndarray:
    off, ln = int(val_idx["offset"][doc_i]), int(val_idx["length"][doc_i])
    return np.asarray(val_mm[off : off + ln])


@torch.no_grad()
def forward_bytes(x: torch.Tensor, lang: torch.Tensor, style: torch.Tensor, *, cq: bool = False, bf16=None) -> dict:
    # One full-stack causal forward on a rectangular [B, T] byte batch -> per-byte NLL (nats) and segmentation.
    kw = dict(raw_bytes=x, lang_ids=lang, stage=STAGE, tau_s=0.5, tau_m=0.5, use_broca_cq=cq)
    if "style_ids" in FWD_PARAMS:
        kw["style_ids"] = style
    if "record_traces" in FWD_PARAMS:
        kw["record_traces"] = False
    use_amp = USE_BF16 if bf16 is None else bool(bf16)
    with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=use_amp):
        out = model(**kw)
    B, T = x.shape
    nll = F.cross_entropy(out.pmd.byte_logits.float().reshape(-1, 256), x.reshape(-1).long(), reduction="none").view(B, T)
    return {
        "nll": nll.cpu().numpy().astype(np.float64),
        "spikes": out.sgsc.spikes.cpu().numpy().astype(bool),
        "p_end": out.pmd.p_seg_end.float().cpu().numpy(),
        "info": out.sgsc.info.float().cpu().numpy().astype(np.float64),
        "loss_lang": float(out.loss_lang),
        "avg_ponder": float(out.metrics.get("avg_ponder_steps", 0.0)),
        "ponder_skip": float(out.metrics.get("ponder_skip_rate", 0.0)),
    }


def eval_matrix(X: np.ndarray, L: np.ndarray, S: np.ndarray, **kw) -> dict:
    # Run forward_bytes over all rows of X [N, T] in batches of BATCH_DOCS and concatenate.
    parts = []
    for i in range(0, len(X), BATCH_DOCS):
        xb = torch.from_numpy(X[i : i + BATCH_DOCS]).to(DEVICE)
        lb = torch.from_numpy(L[i : i + BATCH_DOCS]).to(DEVICE)
        sb = torch.from_numpy(S[i : i + BATCH_DOCS]).to(DEVICE)
        parts.append(forward_bytes(xb, lb, sb, **kw))
    out = {k: np.concatenate([p[k] for p in parts], axis=0) for k in ("nll", "spikes", "p_end", "info")}
    for k in ("loss_lang", "avg_ponder", "ponder_skip"):
        out[k] = float(np.mean([p[k] for p in parts]))
    return out


def bpb(nll: np.ndarray) -> float:
    return float(nll.mean() / LN2)


@contextmanager
def ablate_ponder(l_max: int):
    # Cap Pondering/MoE iterations (0 = experts never run; z* is the fused shortcut z_0).
    old = cfg.l_max
    cfg.l_max = int(l_max)
    try:
        yield
    finally:
        cfg.l_max = old


@contextmanager
def ablate_concept(mode: str):
    # Replace the per-segment concept drive fed to the byte decoder.
    #   mean_both / mean_f / mean_d : replace f_out and/or delta_syn by their mean over all segments
    #   shuffle_both                : give every segment another segment's (f_out, delta_syn)
    orig = model.pmd.decode_teacher_forced
    gen = torch.Generator(device="cpu").manual_seed(SEED)

    def patched(*args, **kwargs):
        ba = _DECODE_SIG.bind(*args, **kwargs)
        f, d = ba.arguments["f_out_seg"], ba.arguments["delta_syn_seg"]
        if f.shape[0] > 1:
            if mode in ("mean_both", "mean_f"):
                f = f.mean(dim=0, keepdim=True).expand_as(f)
            if mode in ("mean_both", "mean_d"):
                d = d.mean(dim=0, keepdim=True).expand_as(d)
            if mode == "shuffle_both":
                perm = torch.randperm(f.shape[0], generator=gen).to(f.device)
                f, d = f[perm], d[perm]
        ba.arguments["f_out_seg"], ba.arguments["delta_syn_seg"] = f, d
        return orig(*ba.args, **ba.kwargs)

    model.pmd.decode_teacher_forced = patched
    try:
        yield
    finally:
        del model.pmd.decode_teacher_forced


# ---- Build the main held-out evaluation matrix: first T bytes of EVAL_DOCS documents ----
T_EVAL = None
for t_try in (EVAL_T, 1024, 512, 256):
    if t_try <= EVAL_T and len(pick_docs(t_try, EVAL_DOCS, SEED)) >= min(EVAL_DOCS, 2 * BATCH_DOCS):
        T_EVAL = t_try
        break
if T_EVAL is None:
    raise RuntimeError("val.idx has too few documents >= 256 bytes for a rectangular evaluation batch.")
docs_main = pick_docs(T_EVAL, EVAL_DOCS, SEED)
X_MAIN = np.stack([doc_bytes(int(d))[:T_EVAL] for d in docs_main]).astype(np.uint8)
L_MAIN = val_idx["lang_id"][docs_main].astype(np.int64)
S_MAIN = val_idx["style_id"][docs_main].astype(np.int64)

t0 = time.perf_counter()
BASE = eval_matrix(X_MAIN, L_MAIN, S_MAIN)
dt_fwd = (time.perf_counter() - t0) / max(1, int(np.ceil(len(X_MAIN) / BATCH_DOCS)))
RESULTS["baseline"] = {
    "bpb": bpb(BASE["nll"]), "docs": int(len(X_MAIN)), "T": int(T_EVAL), "bytes": int(X_MAIN.size),
    "sgsc_prior_bpb": float(BASE["info"].mean() / LN2),
    "mean_seg_len": float(BASE["spikes"].size / max(1, BASE["spikes"].sum())),
    "avg_ponder_steps": BASE["avg_ponder"], "ponder_skip_rate": BASE["ponder_skip"],
    "per_lang_bpb": {int(l): bpb(BASE["nll"][L_MAIN == l]) for l in np.unique(L_MAIN)},
}
b = RESULTS["baseline"]
print(f"[Baseline] {b['docs']} docs x {b['T']} B = {b['bytes']:,} held-out bytes | {dt_fwd:.1f}s per forward")
print(f"[Baseline] model bpb = {b['bpb']:.4f}   (sanity: model's own loss_lang/ln2 = {BASE['loss_lang'] / LN2:.4f})")
print(f"[Baseline] SGSC sensory prior bpb = {b['sgsc_prior_bpb']:.4f}  (context-free byte predictor inside the model)")
print(f"[Baseline] mean segment length = {b['mean_seg_len']:.2f} B | avg ponder steps = {b['avg_ponder_steps']:.2f} | ponder skip rate = {b['ponder_skip_rate']:.2%}")
print("[Baseline] per lang_id bpb: " + ", ".join(f"{k}: {v:.3f} (n={int((L_MAIN == k).sum())})" for k, v in b["per_lang_bpb"].items()))

In [ ]:
# Cell 6 [测试 A：全栈因果泄漏]
# 把每行 t0 之后的字节全部换成随机字节；t0 之前每个位置的 NLL 必须不变。
# 在 fp32 下跑以排除 bf16 数值噪声。不通过 = bpb 被未来信息污染，后面所有数字都不可信。
LEAK_TOL = 1e-3  # nats


def leak_test(cq: bool) -> dict:
    n = min(BATCH_DOCS, len(X_MAIN))
    x = X_MAIN[:n].copy()
    t0_cut = x.shape[1] // 2
    x2 = x.copy()
    rng = np.random.default_rng(SEED + 7)
    x2[:, t0_cut:] = rng.integers(32, 127, size=x2[:, t0_cut:].shape, dtype=np.uint8)
    args = (torch.from_numpy(L_MAIN[:n]).to(DEVICE), torch.from_numpy(S_MAIN[:n]).to(DEVICE))
    a = forward_bytes(torch.from_numpy(x).to(DEVICE), *args, cq=cq, bf16=False)
    c = forward_bytes(torch.from_numpy(x2).to(DEVICE), *args, cq=cq, bf16=False)
    diff = np.abs(a["nll"][:, :t0_cut] - c["nll"][:, :t0_cut])
    bad = np.nonzero(diff > LEAK_TOL)
    res = {
        "max_abs_nll_diff": float(diff.max()),
        "positions_changed": int(len(bad[0])),
        "positions_checked": int(diff.size),
        "segmentation_changed": bool((a["spikes"][:, :t0_cut] != c["spikes"][:, :t0_cut]).any()),
        "max_lookahead_bytes": int(t0_cut - bad[1].min()) if len(bad[0]) else 0,
    }
    res["pass"] = res["positions_changed"] == 0 and not res["segmentation_changed"]
    return res


RESULTS["leak"] = {"mode1": leak_test(cq=False)}
if STAGE in ("C", "D"):
    RESULTS["leak"]["broca_cq"] = leak_test(cq=True)
for name, r in RESULTS["leak"].items():
    if r["pass"]:
        verdict = "PASS (causal)"
    elif STAGE == "A":
        verdict = "FAIL as expected (Stage A auto-encodes each segment from its own bytes; this is the positive control)"
    else:
        verdict = "FAIL (future bytes leak into earlier predictions!)"
    print(
        f"[Leak][{name}] {verdict} | max |dNLL| before cut = {r['max_abs_nll_diff']:.2e} nats | "
        f"changed {r['positions_changed']}/{r['positions_checked']} positions | "
        f"earliest affected position is {r['max_lookahead_bytes']} B before the cut"
    )

In [ ]:
# Cell 7 [测试 B：路径消融 —— MoE 专家与概念路径各贡献多少 bpb]
ABL: dict[str, float] = {"baseline": bpb(BASE["nll"])}
ABL_INFO: dict[str, dict] = {}

if STAGE != "A":
    for lm in sorted({0, 1} & set(range(cfg.l_max))):
        with ablate_ponder(lm):
            r = eval_matrix(X_MAIN, L_MAIN, S_MAIN)
        ABL[f"ponder_l_max={lm}"] = bpb(r["nll"])

for mode in ("mean_both", "mean_f", "mean_d", "shuffle_both"):
    with ablate_concept(mode):
        r = eval_matrix(X_MAIN, L_MAIN, S_MAIN)
    ABL[f"concept_{mode}"] = bpb(r["nll"])

RESULTS["ablation"] = {k: {"bpb": v, "delta_vs_baseline": v - ABL["baseline"]} for k, v in ABL.items()}
_desc = {
    "baseline": "完整模型",
    "ponder_l_max=0": "关闭 Pondering（MoE 专家完全不运行）",
    "ponder_l_max=1": "Pondering 最多 1 步",
    "concept_mean_both": "f_out 与 delta_syn 都换成全局均值（解码器看不到任何概念内容）",
    "concept_mean_f": "只把 f_out（词干通道）换成均值",
    "concept_mean_d": "只把 delta_syn（句法通道）换成均值",
    "concept_shuffle_both": "把别的段的概念驱动喂给当前段",
}
print(f"{'configuration':24s} {'bpb':>8s} {'delta':>8s}   meaning")
for k, v in ABL.items():
    print(f"{k:24s} {v:8.4f} {v - ABL['baseline']:+8.4f}   {_desc.get(k, '')}")

if "ponder_l_max=0" in ABL:
    d_moe = ABL["ponder_l_max=0"] - ABL["baseline"]
    share = 100.0 * RESULTS["params_moe_experts"] / RESULTS["params_total"]
    if abs(d_moe) < 0.01:
        print(f"\n[Verdict B1] 关闭 MoE 只改变 {d_moe:+.4f} bpb：占 {share:.0f}% 参数的专家对语言建模损失基本没有贡献。")
    elif d_moe < 0:
        print(f"\n[Verdict B1] 关闭 MoE 后 bpb 反而下降 {-d_moe:.4f}：专家在干扰预测。")
    else:
        print(f"\n[Verdict B1] 关闭 MoE 使 bpb 上升 {d_moe:.4f}：专家有贡献（占 {share:.0f}% 参数）。")
d_con = ABL["concept_mean_both"] - ABL["baseline"]
print(
    f"[Verdict B2] 概念路径（8 槽位图 -> c_hat -> 解码器）总贡献 = {d_con:.4f} bpb；"
    f"去掉后剩下的 {ABL['concept_mean_both']:.4f} bpb 是字节运动核单独的能力。"
)

In [ ]:
# Cell 8 [测试 C：上下文利用曲线 —— 给 N 字节历史时目标窗口的 bpb]
# 同一批文档、同一个目标窗口（CTX_WINDOW 字节），只改变窗口前面喂给模型的历史长度 N。
# 曲线在哪里走平，模型的有效上下文就有多长。
W = int(CTX_WINDOW)
N_LIST = None
for n_max in (1792, 768, 256):
    if n_max + W <= max(EVAL_T, 512) and len(pick_docs(n_max + W + 4, 2 * BATCH_DOCS, SEED + 1)) >= BATCH_DOCS:
        N_LIST = [n for n in (0, 16, 32, 64, 128, 256, 512, 1024, 1792) if n <= n_max]
        break

if N_LIST is None:
    print("[Context] val documents are too short for a context sweep; skipped.")
    RESULTS["context"] = None
else:
    n_max = N_LIST[-1]
    docs_ctx = pick_docs(n_max + W + 4, 2 * BATCH_DOCS, SEED + 1)
    bufs = [doc_bytes(int(d)) for d in docs_ctx]
    starts = [utf8_start(buf, n_max) for buf in bufs]  # target window starts on a character boundary
    L_CTX = val_idx["lang_id"][docs_ctx].astype(np.int64)
    S_CTX = val_idx["style_id"][docs_ctx].astype(np.int64)
    ctx_curve: dict[int, float] = {}
    for n_ctx in N_LIST:
        Xn = np.stack([buf[s - n_ctx : s + W] for buf, s in zip(bufs, starts)]).astype(np.uint8)
        r = eval_matrix(Xn, L_CTX, S_CTX)
        ctx_curve[n_ctx] = bpb(r["nll"][:, n_ctx:])
        print(f"[Context] history = {n_ctx:5d} B -> target-window bpb = {ctx_curve[n_ctx]:.4f}")
    best = ctx_curve[n_max]
    eff = next(n for n in N_LIST if ctx_curve[n] - best <= 0.02)
    RESULTS["context"] = {
        "window": W, "docs": int(len(docs_ctx)), "curve": {int(k): v for k, v in ctx_curve.items()},
        "effective_context_bytes": int(eff), "gain_total_bpb": ctx_curve[0] - best,
    }
    if ctx_curve[0] - best < 0.02:
        print(f"\n[Verdict C] 从 0 到 {n_max} 字节历史只改变 {ctx_curve[0] - best:+.4f} bpb：模型基本没有利用窗口之前的历史。")
    else:
        print(
            f"\n[Verdict C] 从 0 到 {n_max} 字节历史共降低 {ctx_curve[0] - best:.4f} bpb；"
            f"历史 >= {eff} 字节之后再加上下文的收益不足 0.02 bpb（有效上下文 ≈ {eff} 字节，"
            f"约 {eff / max(RESULTS['baseline']['mean_seg_len'], 1e-6):.0f} 个概念段）。"
        )
    try:
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(6, 3.5))
        ax.plot([max(n, 8) for n in N_LIST], [ctx_curve[n] for n in N_LIST], marker="o")
        ax.set_xscale("log", base=2)
        ax.set_xlabel("history bytes given to the model (0 plotted at 8)")
        ax.set_ylabel("bpb on target window")
        ax.set_title("Context utilisation")
        ax.grid(alpha=0.3)
        plt.show()
    except Exception as exc:
        print(f"[Context] plot skipped: {exc}")

In [ ]:
# Cell 9 [测试 D：段内位置难度 + 运动端/感知端分段一致性]
nll, spk = BASE["nll"], BASE["spikes"]
Bn, Tn = nll.shape
pos = np.broadcast_to(np.arange(Tn), (Bn, Tn))
seg_start = np.concatenate([np.ones((Bn, 1), dtype=bool), spk[:, :-1]], axis=1)
last_start = np.maximum.accumulate(np.where(seg_start, pos, 0), axis=1)
tau = pos - last_start + 1  # 1-based byte offset inside its segment

by_tau = {int(t): bpb(nll[tau == t]) for t in range(1, cfg.max_seg_len + 1) if (tau == t).sum() >= 50}
bpb_last, bpb_first = bpb(nll[spk]), bpb(nll[seg_start])
bpb_mid = bpb(nll[~spk & ~seg_start]) if (~spk & ~seg_start).any() else float("nan")
share_last = float(nll[spk].sum() / nll.sum())

pred_end = BASE["p_end"] >= 0.5
tp = float((pred_end & spk).sum())
prec, rec = tp / max(1.0, float(pred_end.sum())), tp / max(1.0, float(spk.sum()))
f1 = 2 * prec * rec / max(1e-9, prec + rec)

RESULTS["segment"] = {
    "bpb_by_tau": by_tau, "bpb_first_byte": bpb_first, "bpb_last_byte": bpb_last, "bpb_middle_bytes": bpb_mid,
    "frac_bytes_last": float(spk.mean()), "frac_loss_on_last": share_last,
    "halt_precision": prec, "halt_recall": rec, "halt_f1": f1,
}
print("[Segment] bpb by position inside segment (tau): " + "  ".join(f"{t}:{v:.2f}" for t, v in by_tau.items()))
print(f"[Segment] first byte = {bpb_first:.3f} | middle bytes = {bpb_mid:.3f} | last byte (the one that triggered the spike) = {bpb_last:.3f} bpb")
print(f"[Verdict D1] 段末字节占全部字节的 {spk.mean():.1%}，却占总损失的 {share_last:.1%}。")
print(
    f"[Verdict D2] 运动端 p_seg_end>=0.5 对感知端 SGSC 脉冲：precision={prec:.3f} recall={rec:.3f} F1={f1:.3f}。"
    "生成时由 p_seg_end 决定何时取下一个概念，F1 离 1 越远，推理时的分段节奏与训练差得越多。"
)

In [ ]:
# Cell 10 [测试 E：参照梯子 —— n-gram 与通用压缩器（不需要训练任何神经网络）]
import bz2
import lzma
import zlib


def ngram_ladder(train: np.ndarray, X: np.ndarray, max_ctx: int = 5) -> dict[int, float]:
    # Interpolated Witten-Bell byte n-gram, context length 0..max_ctx, counted on `train`, scored on rows of X.
    p = np.full(X.shape, 1.0 / 256.0, dtype=np.float64)
    out: dict[int, float] = {}
    tr = train.astype(np.uint64)
    n = len(tr)
    Bx, Tx = X.shape
    for k in range(0, max_ctx + 1):
        ctx_tr = np.zeros(n - k, dtype=np.uint64)
        for i in range(k):
            ctx_tr = (ctx_tr << np.uint64(8)) | tr[i : n - k + i]
        joint_tr = (ctx_tr << np.uint64(8)) | tr[k:]
        uj, cj = np.unique(joint_tr, return_counts=True)
        del ctx_tr, joint_tr
        uc, first_idx, n1 = np.unique(uj >> np.uint64(8), return_index=True, return_counts=True)
        cc = np.add.reduceat(cj, first_idx)

        Xp = np.concatenate([np.zeros((Bx, k), dtype=np.uint64), X.astype(np.uint64)], axis=1)
        ctx_v = np.zeros((Bx, Tx), dtype=np.uint64)
        for i in range(k):
            ctx_v = (ctx_v << np.uint64(8)) | Xp[:, i : i + Tx]
        joint_v = (ctx_v << np.uint64(8)) | Xp[:, k:]

        pc = np.minimum(np.searchsorted(uc, ctx_v), len(uc) - 1)
        hit_c = uc[pc] == ctx_v
        c_ctx = np.where(hit_c, cc[pc], 0).astype(np.float64)
        n1_v = np.where(hit_c, n1[pc], 0).astype(np.float64)
        pj = np.minimum(np.searchsorted(uj, joint_v), len(uj) - 1)
        c_joint = np.where(uj[pj] == joint_v, cj[pj], 0).astype(np.float64)
        denom = c_ctx + n1_v
        p = np.where(denom > 0, (c_joint + n1_v * p) / np.maximum(denom, 1.0), p)
        out[k] = float(-np.log2(p).mean())
    return out


LADDER: dict[str, float] = {}
t0 = time.perf_counter()
if NGRAM_SAMPLE.exists() and NGRAM_SAMPLE.stat().st_size > 0:
    train_sample = np.fromfile(NGRAM_SAMPLE, dtype=np.uint8)
    for k, v in ngram_ladder(train_sample, X_MAIN, max_ctx=5).items():
        LADDER[f"{k + 1}-gram (context {k} B, {len(train_sample) / 1e6:.0f} MB counts)"] = v
    print(f"[Ladder] n-grams scored on the exact same {X_MAIN.size:,} bytes as the model in {time.perf_counter() - t0:.1f}s")
else:
    print("[Ladder] no train sample available; n-gram rungs skipped.")

comp_raw = bytes(val_mm[: min(len(val_mm), int(COMPRESS_MB) << 20)])
t0 = time.perf_counter()
LADDER["zlib -9 (gzip)"] = 8.0 * len(zlib.compress(comp_raw, 9)) / len(comp_raw)
LADDER["bz2 -9"] = 8.0 * len(bz2.compress(comp_raw, 9)) / len(comp_raw)
LADDER["xz -9e (lzma)"] = 8.0 * len(lzma.compress(comp_raw, preset=9 | lzma.PRESET_EXTREME)) / len(comp_raw)
try:
    import zstandard

    LADDER["zstd -19"] = 8.0 * len(zstandard.ZstdCompressor(level=19).compress(comp_raw)) / len(comp_raw)
except ImportError:
    pass
print(f"[Ladder] compressors run on the first {len(comp_raw) / 1e6:.1f} MB of val.bin in {time.perf_counter() - t0:.1f}s")

LADDER["DCPA byte motor core alone (concept path ablated)"] = ABL["concept_mean_both"]
LADDER["DCPA SGSC sensory prior"] = RESULTS["baseline"]["sgsc_prior_bpb"]
LADDER[">>> DCPA full model <<<"] = ABL["baseline"]
RESULTS["ladder"] = dict(LADDER)

print("\nbits per byte, lower is better (uniform random = 8.000)")
for name, v in sorted(LADDER.items(), key=lambda kv: -kv[1]):
    print(f"  {v:7.4f}   {name}")

In [ ]:
# Cell 11 [汇总与保存]
b = RESULTS["baseline"]
lad = RESULTS["ladder"]
ngram_rungs = sorted((v, k) for k, v in lad.items() if "-gram" in k)
print("=" * 78)
print(f"DCPA diagnosis | {RESULTS['ckpt']} | step {RESULTS['step']} | eval stage {RESULTS['eval_stage']} | {RESULTS['params_total'] / 1e6:.1f}M params")
print("=" * 78)
leak_ok = all(r["pass"] for r in RESULTS["leak"].values())
if RESULTS["eval_stage"] == "A":
    print("A 因果性      : Stage A 是自编码，bpb 不是语言模型分数（训练进入 Stage 2 之后再跑）")
else:
    print(f"A 因果性      : {'通过' if leak_ok else '未通过 —— 下面的 bpb 被未来信息污染，先修泄漏'}")
print(f"  模型 bpb    : {b['bpb']:.4f}  (held-out, {b['bytes']:,} bytes)")
if "ponder_l_max=0" in RESULTS["ablation"]:
    print(f"B MoE 专家    : 关闭后 {RESULTS['ablation']['ponder_l_max=0']['delta_vs_baseline']:+.4f} bpb  (专家占参数 {100 * RESULTS['params_moe_experts'] / RESULTS['params_total']:.0f}%)")
print(f"B 概念路径    : 去掉后 {RESULTS['ablation']['concept_mean_both']['delta_vs_baseline']:+.4f} bpb")
if RESULTS.get("context"):
    c = RESULTS["context"]
    print(f"C 有效上下文  : ≈ {c['effective_context_bytes']} 字节  (全部历史共带来 {c['gain_total_bpb']:.4f} bpb)")
s = RESULTS["segment"]
print(f"D 段末字节    : {s['bpb_last_byte']:.3f} bpb vs 段中 {s['bpb_middle_bytes']:.3f} bpb；占损失 {s['frac_loss_on_last']:.1%}")
print(f"D 分段一致性  : p_seg_end 对 SGSC 脉冲 F1 = {s['halt_f1']:.3f}")
if ngram_rungs:
    beaten = [k for v, k in ngram_rungs if b["bpb"] < v]
    not_beaten = [k for v, k in ngram_rungs if b["bpb"] >= v]
    print(f"E 对 n-gram   : 优于 {len(beaten)}/{len(ngram_rungs)} 档" + (f"；未超过: {not_beaten[0]} ({lad[not_beaten[0]]:.3f})" if not_beaten else ""))
xz = lad.get("xz -9e (lzma)")
if xz is not None:
    print(f"E 对 xz       : 模型 {b['bpb']:.3f} vs xz {xz:.3f} bpb -> {'优于' if b['bpb'] < xz else '不如'}通用压缩器")
print("=" * 78)

out_dir = Path(os.environ.get("DCPA_DIAG_OUT", str(DRIVE_LOGS)))
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"diag_step_{RESULTS['step']}_{time.strftime('%Y%m%d_%H%M%S')}.json"
out_path.write_text(json.dumps(RESULTS, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print(f"[Diag] Full results saved to {out_path}")